1. Importing Necessary Libraries

In [2]:
import os, glob
from pathlib import Path
import numpy as np
import tensorflow as tf
from tensorflow import keras
from PIL import Image, ImageOps
import cv2
import pandas as pd
from tqdm import tqdm

print("TF:", tf.__version__)

TF: 2.20.0


2. Global configuration for ANN/CNN/RNN adversarial experiments

In [3]:
BASE_DIR        = '.'                                    
CSV_PATH        = './fruit_quality_split_annotations.csv'                

ANN_MODEL_PATH  = './models/fruit_quality_ann.keras'    
CNN_MODEL_PATH  = './models/fruit_quality_cnn.keras'     
RNN_MODEL_PATH  = './models/fruit_quality_rnn.keras'     

IMG_SIZE        = 128     

EPS_FGSM        = 0.10             
EPS_PGD         = 0.10              
PGD_STEPS       = 40               
PGD_STEP_SIZE   = EPS_PGD / 10.0    

ANN_USE_LOGITS  = False
CNN_USE_LOGITS  = False
RNN_USE_LOGITS  = False

SAVE_DIR_ROOT  = r'E:/adv_outputs_fruit_quality' 

In [5]:


OUT_ROOT = r'E:/adv_outputs_fruit_quality' 

if isinstance(OUT_ROOT, str):
    OUT_ROOT = Path(OUT_ROOT)
    
ATTACKS = ["fgsm", "pgd"]

MODEL_TAG = "MODEL"   

g = globals()
model_tags = []
if "ANN_MODEL_PATH" in g and g["ANN_MODEL_PATH"]:
    model_tags.append("ANN")
if "CNN_MODEL_PATH" in g and g["CNN_MODEL_PATH"]:
    model_tags.append("CNN")
if "RNN_MODEL_PATH" in g and g["RNN_MODEL_PATH"]:
    model_tags.append("RNN")

if not model_tags:
    model_tags = [MODEL_TAG]

print("Model tags detected / used:", model_tags)

df_tmp = pd.read_csv(CSV_PATH)
splits = sorted(df_tmp["split"].unique())
unique_labels = sorted(df_tmp["label"].unique())

if len(unique_labels) == 2:
    label_map = {unique_labels[0]: "bad", unique_labels[1]: "good"}
elif len(unique_labels) == 3:
    label_map = {unique_labels[i]: name for i, name in enumerate(["bad","good","mixed"])}
else:
    label_map = {lab: f"class_{lab}" for lab in unique_labels}

print("Splits found:", splits)
print("Label mapping:", label_map)

created = []
for mtag in model_tags:
    for attack in ATTACKS:
        root_folder = OUT_ROOT / f"{mtag}_{attack}"
        for split in splits:
            for lab in label_map.values():
                p = root_folder / split / lab
                p.mkdir(parents=True, exist_ok=True)
                created.append(str(p))

print(f"Created {len(created)} folders under {OUT_ROOT}. (Showing up to 50)")
for c in created[:50]:
    print("  ", c)

Model tags detected / used: ['ANN', 'CNN', 'RNN']
Splits found: ['test', 'train', 'valid']
Label mapping: {0: 'bad', 1: 'good', 2: 'mixed'}
Created 54 folders under E:\adv_outputs_fruit_quality. (Showing up to 50)
   E:\adv_outputs_fruit_quality\ANN_fgsm\test\bad
   E:\adv_outputs_fruit_quality\ANN_fgsm\test\good
   E:\adv_outputs_fruit_quality\ANN_fgsm\test\mixed
   E:\adv_outputs_fruit_quality\ANN_fgsm\train\bad
   E:\adv_outputs_fruit_quality\ANN_fgsm\train\good
   E:\adv_outputs_fruit_quality\ANN_fgsm\train\mixed
   E:\adv_outputs_fruit_quality\ANN_fgsm\valid\bad
   E:\adv_outputs_fruit_quality\ANN_fgsm\valid\good
   E:\adv_outputs_fruit_quality\ANN_fgsm\valid\mixed
   E:\adv_outputs_fruit_quality\ANN_pgd\test\bad
   E:\adv_outputs_fruit_quality\ANN_pgd\test\good
   E:\adv_outputs_fruit_quality\ANN_pgd\test\mixed
   E:\adv_outputs_fruit_quality\ANN_pgd\train\bad
   E:\adv_outputs_fruit_quality\ANN_pgd\train\good
   E:\adv_outputs_fruit_quality\ANN_pgd\train\mixed
   E:\adv_outputs_

In [ ]:
from pathlib import Path
import pandas as pd

df = pd.read_csv(CSV_PATH)

df["image_id_norm"] = df["image_id"].astype(str).str.replace("\\\\", "/", regex=True).str.lower()

image_to_split = dict(zip(df["image_id_norm"], df["split"]))
image_to_label = dict(zip(df["image_id_norm"], df["label"]))

label_map = {0: "good", 1: "bad", 2: "mixed"}

def get_split_label(path):
    key = str(path).replace("\\", "/").lower()
    split = image_to_split.get(key, "test")
    label_val = image_to_label.get(key, None)
    label_name = label_map.get(label_val, f"class_{label_val}" if label_val is not None else "unknown")
    return split, label_name

3. Robust CSV loading and path utilities (OS-agnostic, mirror-save helpers)

In [7]:
def read_csv_robust(path):

    for sep in [None, ',', ';', '\t']:
        for enc in ['utf-8-sig', 'utf-8', 'cp1252']:
            try:
                df = pd.read_csv(path, sep=sep, engine='python', encoding=enc)
                if df.shape[1] >= 2:
                    return df
            except Exception:
                pass
    raise AssertionError(f'Failed to read CSV: {path}')

def resolve_path(image_id):

    p = str(image_id).replace('\\', os.sep).replace('/', os.sep)
    if os.path.isabs(p): return p
    return os.path.normpath(os.path.join(BASE_DIR, p))

def mirror_out_path(save_root, original_path):

    try:
        rel = os.path.relpath(original_path, start=BASE_DIR)
    except ValueError:
        rel = os.path.basename(original_path)

    rel = rel.replace('\\', os.sep).replace('/', os.sep)
    out_path = os.path.join(save_root, os.path.splitext(rel)[0] + '.png')

    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    return out_path

4. Image preprocessing utilities (RGB normalized & backend-equivalent BGR 50×50)

In [8]:
def preprocess_rgb(path, target=IMG_SIZE):

    with Image.open(path) as pil:
        pil = ImageOps.exif_transpose(pil.convert('RGB'))
        arr = np.array(pil)  

    arr = cv2.resize(arr, (target, target), interpolation=cv2.INTER_AREA)

    return (arr.astype(np.float32) / 255.0)

def preprocess_bgr_50x50(path_or_stream):

    if hasattr(path_or_stream, 'read'):
        pil = Image.open(path_or_stream).convert('RGB')
        pil = ImageOps.exif_transpose(pil)
        arr = np.array(pil)
    else:
        with Image.open(path_or_stream) as pil:
            pil = ImageOps.exif_transpose(pil.convert('RGB'))
            arr = np.array(pil)

    bgr = cv2.cvtColor(arr, cv2.COLOR_RGB2BGR)

    bgr = cv2.resize(bgr, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)

    return (bgr.astype(np.float32) / 255.0)

5. FGSM & PGD for ANN with fixed auxiliary feature (isNight)

In [9]:
def fgsm_ann(model, x_input_np: np.ndarray, eps: float = 0.1, from_logits: bool = False) -> np.ndarray:

    x = tf.convert_to_tensor(x_input_np, dtype=tf.float32)
    with tf.GradientTape() as tape:
        tape.watch(x)
        preds = model(x, training=False)
        y_idx = tf.argmax(preds, axis=1)
        y_true = tf.one_hot(y_idx, depth=int(preds.shape[-1]))
        loss = tf.keras.losses.categorical_crossentropy(y_true, preds, from_logits=from_logits)

    grad = tape.gradient(loss, x)
    signed = tf.sign(grad)

    mask = tf.concat([tf.ones_like(x[:, :-1]), tf.zeros_like(x[:, -1:])], axis=1) 
    signed = signed * mask

    x_adv = x + eps * signed
    x_adv_np = x_adv.numpy()

    x_adv_np[:, -1] = x_input_np[:, -1]
    x_adv_np[:, :-1] = np.clip(x_adv_np[:, :-1], 0.0, 1.0)
    return x_adv_np

def pgd_ann(model, x_input_np: np.ndarray, eps: float = 0.1, step: float = 0.01, steps: int = 40, from_logits: bool=False) -> np.ndarray:

    x0 = tf.convert_to_tensor(x_input_np, dtype=tf.float32)
    x = tf.identity(x0)
    for _ in range(int(steps)):
        with tf.GradientTape() as tape:
            tape.watch(x)
            preds = model(x, training=False)
            y_idx = tf.argmax(preds, axis=1)
            y_true = tf.one_hot(y_idx, depth=int(preds.shape[-1]))
            loss = tf.keras.losses.categorical_crossentropy(y_true, preds, from_logits=from_logits)

        grad = tape.gradient(loss, x)
        signed = tf.sign(grad)

        mask = tf.concat([tf.ones_like(x[:, :-1]), tf.zeros_like(x[:, -1:])], axis=1)
        signed = signed * mask

        x = x + step * signed

        delta = x - x0
        delta_pixels = tf.clip_by_value(delta[:, :-1], -eps, eps)
        x = tf.concat([x0[:, :-1] + delta_pixels, x0[:, -1:]], axis=1)

        x_pixels = tf.clip_by_value(x[:, :-1], 0.0, 1.0)
        x = tf.concat([x_pixels, x[:, -1:]], axis=1)

    return x.numpy()

def vector_to_rgb_image(x_row, img_size=IMG_SIZE):

    bgr = x_row[:-1].reshape(img_size, img_size, 3)
    rgb = cv2.cvtColor((bgr * 255).astype('uint8'), cv2.COLOR_BGR2RGB)
    return rgb

6. Pixel-space FGSM & PGD (single-/multi-input) with 𝐿∞ constraint

In [10]:
def fgsm_pixels(model, x_pixels, eps=0.1, from_logits=False, y_true=None):

    x0 = tf.convert_to_tensor(x_pixels, dtype=tf.float32)
    with tf.GradientTape() as tape:
        tape.watch(x0)
        preds = model(x0, training=False)
        if y_true is None:
            y_idx = tf.argmax(preds, axis=1)
            y_true = tf.one_hot(y_idx, depth=int(preds.shape[-1]))
        loss = tf.keras.losses.categorical_crossentropy(y_true, preds, from_logits=from_logits)
    grad = tape.gradient(loss, x0)
    x_adv = x0 + eps * tf.sign(grad)               
    x_adv = tf.clip_by_value(x_adv, 0.0, 1.0)      
    return x_adv.numpy()

def pgd_pixels(model, x_pixels, eps=0.1, step=0.01, steps=40, from_logits=False, y_true=None):

    x0 = tf.convert_to_tensor(x_pixels, dtype=tf.float32)
    x = tf.identity(x0)
    for _ in range(int(steps)):
        with tf.GradientTape() as tape:
            tape.watch(x)
            preds = model(x, training=False)
            if y_true is None:
                y_idx = tf.argmax(preds, axis=1)
                y_true_dyn = tf.one_hot(y_idx, depth=int(preds.shape[-1]))
            else:
                y_true_dyn = y_true
            loss = tf.keras.losses.categorical_crossentropy(y_true_dyn, preds, from_logits=from_logits)
        grad = tape.gradient(loss, x)
        x = x + step * tf.sign(grad)    
        x = tf.clip_by_value(tf.minimum(x0 + eps, tf.maximum(x0 - eps, x)), 0.0, 1.0)
    return x.numpy()

def fgsm_pixels_multi(model, x_pixels, aux, eps=0.1, from_logits=False, y_true=None):
    x0 = tf.convert_to_tensor(x_pixels, dtype=tf.float32)
    aux_tf = tf.convert_to_tensor(aux, dtype=tf.float32)
    with tf.GradientTape() as tape:
        tape.watch(x0)
        preds = model([x0, aux_tf], training=False)
        if y_true is None:
            y_idx = tf.argmax(preds, axis=1)
            y_true = tf.one_hot(y_idx, depth=int(preds.shape[-1]))
        loss = tf.keras.losses.categorical_crossentropy(y_true, preds, from_logits=from_logits)
    grad = tape.gradient(loss, x0)
    x_adv = x0 + eps * tf.sign(grad)              
    x_adv = tf.clip_by_value(x_adv, 0.0, 1.0)
    return x_adv.numpy()

def pgd_pixels_multi(model, x_pixels, aux, eps=0.1, step=0.01, steps=40, from_logits=False, y_true=None):
    x0 = tf.convert_to_tensor(x_pixels, dtype=tf.float32)
    aux_tf = tf.convert_to_tensor(aux, dtype=tf.float32)
    x = tf.identity(x0)
    for _ in range(int(steps)):
        with tf.GradientTape() as tape:
            tape.watch(x)
            preds = model([x, aux_tf], training=False)
            if y_true is None:
                y_idx = tf.argmax(preds, axis=1)
                y_true_dyn = tf.one_hot(y_idx, depth=int(preds.shape[-1]))
            else:
                y_true_dyn = y_true
            loss = tf.keras.losses.categorical_crossentropy(y_true_dyn, preds, from_logits=from_logits)
        grad = tape.gradient(loss, x)
        x = x + step * tf.sign(grad)
        x = tf.clip_by_value(tf.minimum(x0 + eps, tf.maximum(x0 - eps, x)), 0.0, 1.0)
    return x.numpy()

7. CNN BGR wrappers (backend-equivalent; optional aux input)

In [11]:
def build_cnn_bgr_wrapper_single(base_model, img_size=IMG_SIZE):
    H=W=img_size
    pix_in = keras.Input(shape=(H,W,3), name='bgr_pixels')
    x = pix_in
    shp = base_model.input_shape
    in_shape = shp if isinstance(shp, (list,tuple)) else (None,) + tuple(shp[1:])
    if len(in_shape)>=4 and in_shape[1] not in (None,-1) and in_shape[2] not in (None,-1):
        tH, tW = int(in_shape[1]), int(in_shape[2])
        if (tH,tW)!=(H,W):
            x = keras.layers.Lambda(lambda t: tf.image.resize(t, (tH,tW)))(x)
    out = base_model(x)  # pass BGR as-is
    return keras.Model(pix_in, out, name=f'cnn_bgr_wrapper_{base_model.name}')

def build_cnn_bgr_wrapper_multi(base_model, img_size=IMG_SIZE, aux_dim=1):
    H=W=img_size
    pix_in = keras.Input(shape=(H,W,3), name='bgr_pixels')
    aux_in = keras.Input(shape=(aux_dim,), name='aux_input')
    x = pix_in
    shp = base_model.input_shape
    in_shape = shp if isinstance(shp, (list,tuple)) else (None,) + tuple(shp[1:])
    img_shape = in_shape[0] if isinstance(in_shape, (list,tuple)) else in_shape
    if len(img_shape)>=4 and img_shape[1] not in (None,-1) and img_shape[2] not in (None,-1):
        tH, tW = int(img_shape[1]), int(img_shape[2])
        if (tH,tW)!=(H,W):
            x = keras.layers.Lambda(lambda t: tf.image.resize(t, (tH,tW)))(x)
    out = base_model([x, aux_in])  
    return keras.Model([pix_in, aux_in], out, name=f'cnn_bgr_wrapper_multi_{base_model.name}')

8. RNN BGR wrappers (reshape 50×50×3 → sequence 50×150, optional aux input)

In [12]:
def build_rnn_bgr_wrapper_single(base_model, img_size=IMG_SIZE):
    H=W=img_size
    pix_in = keras.Input(shape=(H,W,3), name='bgr_pixels')
    seq = keras.layers.Lambda(lambda t: tf.reshape(t, (-1, H, W*3)))(pix_in)  
    out = base_model(seq)
    return keras.Model(pix_in, out, name='rnn_bgr_wrapper_single')

def build_rnn_bgr_wrapper_multi(base_model, img_size=IMG_SIZE, aux_dim=1):

    H=W=img_size
    pix_in = keras.Input(shape=(H,W,3), name='bgr_pixels')
    aux_in = keras.Input(shape=(aux_dim,), name='aux_input')

    seq = keras.layers.Lambda(lambda t: tf.reshape(t, (-1, H, W*3)))(pix_in)
    out = base_model([seq, aux_in])
    return keras.Model([pix_in, aux_in], out, name='rnn_bgr_wrapper_multi')

9. Build dataset items from CSV (paths, labels, optional isNight)

In [13]:
df = read_csv_robust(CSV_PATH)

cols = {c.lower(): c for c in df.columns}

col_image = cols.get('image_id') or cols.get('image') or list(df.columns)[0]
col_label = cols.get('label') or list(df.columns)[1]
col_night = cols.get('isnight')

paths  = [resolve_path(p) for p in df[col_image].tolist()]

labels = df[col_label].astype(int).tolist()
nights = df[col_night].astype(float).tolist() if col_night is not None else [0.0]*len(paths)

items = list(zip(paths, labels, nights))

print(f"Loaded {len(items)} items. Example:", items[:2])

Loaded 19526 items. Example: [('E:\\Fruit_Quality_Split\\train\\bad\\IMG_2510.JPG', 1, 1.0), ('E:\\Fruit_Quality_Split\\train\\good\\IMG20200728182302_27596.jpg', 0, 1.0)]


10. Load models and construct wrappers (ANN / CNN / RNN)

In [14]:
models = {}

if os.path.exists(ANN_MODEL_PATH):
    models['ANN'] = keras.models.load_model(ANN_MODEL_PATH)
    print("Loaded ANN:", models['ANN'].input_shape, "->", models['ANN'].output_shape)
else:
    print("[WARN] ANN model not found:", ANN_MODEL_PATH)

if os.path.exists(CNN_MODEL_PATH):
    models['CNN_base'] = keras.models.load_model(CNN_MODEL_PATH)
    print("Loaded CNN base:", models['CNN_base'].input_shape, "->", models['CNN_base'].output_shape)

    base_in_shape = models['CNN_base'].input_shape
    if isinstance(base_in_shape, (list, tuple)) and len(base_in_shape) >= 2:
        aux_shape = base_in_shape[1]       
        aux_dim = 1
        try:
            if aux_shape is not None and hasattr(aux_shape, '__len__') and len(aux_shape) > 0 and aux_shape[-1] not in (None, -1):
                aux_dim = int(aux_shape[-1])
        except Exception:
            aux_dim = 1

        models['CNN'] = build_cnn_bgr_wrapper_multi(models['CNN_base'], IMG_SIZE, aux_dim=aux_dim)
        models['CNN_needs_aux'] = True
        print("Built CNN BGR multi wrapper; aux_dim=", aux_dim)
    else:
        models['CNN'] = build_cnn_bgr_wrapper_single(models['CNN_base'], IMG_SIZE)
        models['CNN_needs_aux'] = False
        print("Built CNN BGR single wrapper")
else:
    print("[WARN] CNN model not found:", CNN_MODEL_PATH)

if os.path.exists(RNN_MODEL_PATH):
    models['RNN_base'] = keras.models.load_model(RNN_MODEL_PATH)
    print("Loaded RNN base:", models['RNN_base'].input_shape, "->", models['RNN_base'].output_shape)

    base_in_shape = models['RNN_base'].input_shape
    if isinstance(base_in_shape, (list, tuple)) and len(base_in_shape) >= 2:
        aux_shape = base_in_shape[1]
        aux_dim = 1
        try:
            if aux_shape is not None and hasattr(aux_shape, '__len__') and len(aux_shape) > 0 and aux_shape[-1] not in (None, -1):
                aux_dim = int(aux_shape[-1])
        except Exception:
            aux_dim = 1

        models['RNN'] = build_rnn_bgr_wrapper_multi(models['RNN_base'], IMG_SIZE, aux_dim=aux_dim)
        models['RNN_needs_aux'] = True
        print("Built RNN BGR multi wrapper; aux_dim=", aux_dim)
    else:
        models['RNN'] = build_rnn_bgr_wrapper_single(models['RNN_base'], IMG_SIZE)
        models['RNN_needs_aux'] = False
        print("Built RNN BGR single wrapper")
else:
    print("[WARN] RNN model not found:", RNN_MODEL_PATH)

Loaded ANN: (None, 49153) -> (None, 3)
Loaded CNN base: [(None, 128, 128, 3), (None, 1)] -> (None, 3)
Built CNN BGR multi wrapper; aux_dim= 1
Loaded RNN base: [(None, 128, 384), (None, 1)] -> (None, 3)

Built RNN BGR multi wrapper; aux_dim= 1


11. Run ANN attacks over dataset and save adversarial PNGs

In [15]:
from pathlib import Path

def mirror_out_path_structured(out_root, model_tag, attack_name, path, split_name, label_name):
    out_dir = Path(out_root) / f"{model_tag}_{attack_name}" / split_name / label_name
    out_dir.mkdir(parents=True, exist_ok=True)
    return out_dir / (Path(path).stem + ".png")

In [ ]:


if 'ANN' in models:
    ann = models['ANN']

    out_root_fgsm = os.path.join(SAVE_DIR_ROOT, 'ANN_fgsm')
    out_root_pgd  = os.path.join(SAVE_DIR_ROOT, 'ANN_pgd')
    os.makedirs(out_root_fgsm, exist_ok=True)
    os.makedirs(out_root_pgd,  exist_ok=True)

    clean_correct = adv_fgsm_correct = adv_pgd_correct = total = 0

    for path, label, is_night in tqdm(items, desc='ANN attacks', unit='img'):
        try:
            bgr = preprocess_bgr_50x50(path)
        except Exception as e:
            print("[WARN] cannot load:", path, "->", e)
            continue

        x_flat = bgr.reshape(1, -1)                                       
        night  = np.array([[float(is_night)]], dtype=np.float32)            
        x      = np.hstack([x_flat, night]).astype(np.float32)              

        clean_idx = int(np.argmax(ann.predict(x, verbose=0)[0]))
        clean_correct += int(clean_idx == label)

        x_adv_fgsm = fgsm_ann(ann, x, eps=EPS_FGSM, from_logits=ANN_USE_LOGITS)

        adv_small = Image.fromarray(vector_to_rgb_image(x_adv_fgsm[0], IMG_SIZE))
        orig = Image.open(path).convert("RGB")
        adv_orig = adv_small.resize(orig.size, resample=Image.BILINEAR)

        split_name, label_name = get_split_label(path)
        save_path = mirror_out_path_structured(SAVE_DIR_ROOT, "ANN", "fgsm", path, split_name, label_name)
        adv_orig.save(save_path)

        idx_fgsm = int(np.argmax(ann.predict(x_adv_fgsm, verbose=0)[0]))
        adv_fgsm_correct += int(idx_fgsm == label)

        x_adv_pgd = pgd_ann(ann, x, eps=EPS_PGD, step=PGD_STEP_SIZE, steps=PGD_STEPS, from_logits=ANN_USE_LOGITS)

        adv_small = Image.fromarray(vector_to_rgb_image(x_adv_pgd[0], IMG_SIZE))
        orig = Image.open(path).convert("RGB")
        adv_orig = adv_small.resize(orig.size, resample=Image.BILINEAR)

        split_name, label_name = get_split_label(path)
        save_path = mirror_out_path_structured(SAVE_DIR_ROOT, "ANN", "pgd", path, split_name, label_name)
        adv_orig.save(save_path)

        idx_pgd = int(np.argmax(ann.predict(x_adv_pgd, verbose=0)[0]))
        adv_pgd_correct += int(idx_pgd == label)

        total += 1

    print(f"[ANN] clean_acc={clean_correct/max(1,total):.4f}  fgsm_acc={adv_fgsm_correct/max(1,total):.4f}  pgd_acc={adv_pgd_correct/max(1,total):.4f}")
else:
    print("Skipping ANN.")

ANN attacks: 100%|██████████| 19526/19526 [4:31:36<00:00,  1.20img/s]  

[ANN] clean_acc=0.9224  fgsm_acc=0.0645  pgd_acc=0.4729


12. Run CNN attacks over dataset and save adversarial PNGs

In [ ]:
if 'CNN' in models:
    cnn = models['CNN']

    out_root_fgsm = os.path.join(SAVE_DIR_ROOT, 'CNN_fgsm')
    out_root_pgd  = os.path.join(SAVE_DIR_ROOT, 'CNN_pgd')
    os.makedirs(out_root_fgsm, exist_ok=True)
    os.makedirs(out_root_pgd,  exist_ok=True)

    clean_correct = adv_fgsm_correct = adv_pgd_correct = total = 0

    for path, label, is_night in tqdm(items, desc='CNN attacks (backend-equiv BGR)', unit='img'):
        try:
            bgr = preprocess_bgr_50x50(path)  
        except Exception as e:
            print("[WARN] cannot load:", path, "->", e)
            continue

        x = bgr[np.newaxis, ...]    
        needs_aux = bool(models.get('CNN_needs_aux'))
        if needs_aux:
            aux = np.array([[float(is_night)]], dtype=np.float32)

        if needs_aux:
            clean_pred = cnn.predict([x, aux], verbose=0)[0]
        else:
            clean_pred = cnn.predict(x, verbose=0)[0]
        clean_idx = int(np.argmax(clean_pred))
        clean_correct += int(clean_idx == label)

        if needs_aux:
            x_adv_fgsm = fgsm_pixels_multi(cnn, x, aux, eps=EPS_FGSM, from_logits=CNN_USE_LOGITS)
            adv_pred_fgsm = cnn.predict([x_adv_fgsm, aux], verbose=0)[0]
        else:
            x_adv_fgsm = fgsm_pixels(cnn, x, eps=EPS_FGSM, from_logits=CNN_USE_LOGITS)
            adv_pred_fgsm = cnn.predict(x_adv_fgsm, verbose=0)[0]

        rgb_png = cv2.cvtColor((x_adv_fgsm[0]*255).astype('uint8'), cv2.COLOR_BGR2RGB)
        orig = Image.open(path).convert("RGB")
        adv_small = Image.fromarray(rgb_png)  
        adv_orig = adv_small.resize(orig.size, resample=Image.BILINEAR)

        split_name, label_name = get_split_label(path)
        save_path = mirror_out_path_structured(SAVE_DIR_ROOT, "CNN", "fgsm", path, split_name, label_name)
        adv_orig.save(save_path)


        idx_fgsm = int(np.argmax(adv_pred_fgsm))
        adv_fgsm_correct += int(idx_fgsm == label)

        if needs_aux:
            x_adv_pgd = pgd_pixels_multi(cnn, x, aux, eps=EPS_PGD, step=PGD_STEP_SIZE, steps=PGD_STEPS, from_logits=CNN_USE_LOGITS)
            adv_pred_pgd = cnn.predict([x_adv_pgd, aux], verbose=0)[0]
        else:
            x_adv_pgd = pgd_pixels(cnn, x, eps=EPS_PGD, step=PGD_STEP_SIZE, steps=PGD_STEPS, from_logits=CNN_USE_LOGITS)
            adv_pred_pgd = cnn.predict(x_adv_pgd, verbose=0)[0]

            
        rgb_png = cv2.cvtColor((x_adv_pgd[0]*255).astype('uint8'), cv2.COLOR_BGR2RGB)
        orig = Image.open(path).convert("RGB")
        adv_small = Image.fromarray(rgb_png)
        adv_orig = adv_small.resize(orig.size, resample=Image.BILINEAR)

        split_name, label_name = get_split_label(path)
        save_path = mirror_out_path_structured(SAVE_DIR_ROOT, "CNN", "pgd", path, split_name, label_name)
        adv_orig.save(save_path)


        idx_pgd = int(np.argmax(adv_pred_pgd))
        adv_pgd_correct += int(idx_pgd == label)

        total += 1

    print(f"[CNN] clean_acc={clean_correct/max(1,total):.4f}  fgsm_acc={adv_fgsm_correct/max(1,total):.4f}  pgd_acc={adv_pgd_correct/max(1,total):.4f}")
else:
    print("Skipping CNN.")

CNN attacks (backend-equiv BGR):   0%|          | 0/19526 [00:00<?, ?img/s]

CNN attacks (backend-equiv BGR): 100%|██████████| 19526/19526 [9:08:20<00:00,  1.68s/img]   

[CNN] clean_acc=0.9657  fgsm_acc=0.2994  pgd_acc=0.2452


13. Generate adversarials only for test set

In [16]:
test_rows = df[df['split']=='test'][['image_id','label','isNight']].dropna()
items_test = [(str(r['image_id']), int(r['label']), int(r['isNight'])) for _, r in test_rows.iterrows()]

print(f"Will generate adversarials for {len(items_test)} test images.")

Will generate adversarials for 1953 test images.


14. Run RNN attacks over dataset and save adversarial PNGs

In [ ]:
if 'RNN' in models:
    rnn = models['RNN']

    out_root_fgsm = os.path.join(SAVE_DIR_ROOT, 'RNN_fgsm')
    out_root_pgd  = os.path.join(SAVE_DIR_ROOT, 'RNN_pgd')
    os.makedirs(out_root_fgsm, exist_ok=True)
    os.makedirs(out_root_pgd,  exist_ok=True)

    clean_correct = adv_fgsm_correct = adv_pgd_correct = total = 0

    for path, label, is_night in tqdm(items_test, desc='RNN attacks (backend-equiv)', unit='img'):
        try:
            bgr = preprocess_bgr_50x50(path)  
        except Exception as e:
            print("[WARN] cannot load:", path, "->", e)
            continue

        x = bgr[np.newaxis, ...]
        needs_aux = bool(models.get('RNN_needs_aux'))
        if needs_aux:
            aux = np.array([[float(is_night)]], dtype=np.float32)

        if needs_aux:
            clean_pred = rnn.predict([x, aux], verbose=0)[0]
        else:
            clean_pred = rnn.predict(x, verbose=0)[0]
        clean_idx = int(np.argmax(clean_pred))
        clean_correct += int(clean_idx == label)

        if needs_aux:
            x_adv_fgsm = fgsm_pixels_multi(rnn, x, aux, eps=EPS_FGSM, from_logits=RNN_USE_LOGITS)
            adv_pred_fgsm = rnn.predict([x_adv_fgsm, aux], verbose=0)[0]
        else:
            x_adv_fgsm = fgsm_pixels(rnn, x, eps=EPS_FGSM, from_logits=RNN_USE_LOGITS)
            adv_pred_fgsm = rnn.predict(x_adv_fgsm, verbose=0)[0]

        rgb_png = cv2.cvtColor((x_adv_fgsm[0] * 255).astype('uint8'), cv2.COLOR_BGR2RGB)
        adv_small = Image.fromarray(rgb_png)                    
        orig = Image.open(path).convert("RGB")               
        adv_orig = adv_small.resize(orig.size, resample=Image.BILINEAR)

        split_name, label_name = get_split_label(path)         
        save_path = mirror_out_path_structured(SAVE_DIR_ROOT, "RNN", "fgsm", path, split_name, label_name)
        adv_orig.save(save_path)


        idx_fgsm = int(np.argmax(adv_pred_fgsm))
        adv_fgsm_correct += int(idx_fgsm == label)

        if needs_aux:
            x_adv_pgd = pgd_pixels_multi(rnn, x, aux, eps=EPS_PGD, step=PGD_STEP_SIZE, steps=PGD_STEPS, from_logits=RNN_USE_LOGITS)
            adv_pred_pgd = rnn.predict([x_adv_pgd, aux], verbose=0)[0]
        else:
            x_adv_pgd = pgd_pixels(rnn, x, eps=EPS_PGD, step=PGD_STEP_SIZE, steps=PGD_STEPS, from_logits=RNN_USE_LOGITS)
            adv_pred_pgd = rnn.predict(x_adv_pgd, verbose=0)[0]

        rgb_png = cv2.cvtColor((x_adv_pgd[0] * 255).astype('uint8'), cv2.COLOR_BGR2RGB)
        adv_small = Image.fromarray(rgb_png)
        orig = Image.open(path).convert("RGB")
        adv_orig = adv_small.resize(orig.size, resample=Image.BILINEAR)

        split_name, label_name = get_split_label(path)
        save_path = mirror_out_path_structured(SAVE_DIR_ROOT, "RNN", "pgd", path, split_name, label_name)
        adv_orig.save(save_path)


        idx_pgd = int(np.argmax(adv_pred_pgd))
        adv_pgd_correct += int(idx_pgd == label)

        total += 1
        
    print(f"[RNN] clean_acc={clean_correct/max(1,total):.4f}  fgsm_acc={adv_fgsm_correct/max(1,total):.4f}  pgd_acc={adv_pgd_correct/max(1,total):.4f}")
else:
    print("Skipping RNN.")

RNN attacks (backend-equiv): 100%|██████████| 1953/1953 [13:16:52<00:00, 24.48s/img]  

[RNN] clean_acc=0.9483  fgsm_acc=0.1807  pgd_acc=0.4706


15. Sanity check — count saved adversarial PNG files

In [ ]:
def count_pngs(root):
    return len(glob.glob(os.path.join(root, '**', '*.png'), recursive=True))

for sub in ['ANN_fgsm','ANN_pgd','CNN_fgsm','CNN_pgd','RNN_fgsm','RNN_pgd']:
    p = os.path.join(SAVE_DIR_ROOT, sub)
    print(sub, count_pngs(p), "files")

ANN_fgsm 19128 files
ANN_pgd 19128 files
CNN_fgsm 19128 files
CNN_pgd 19128 files
RNN_fgsm 1947 files
RNN_pgd 1947 files
